# The Bias-Variance Trade-off & Cross Validation
## DS-3001: Foundations of Machine Learning

Content adapted from Terence Johnson (UVA)

### Loading Environment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os # For changing directory

# To mount your google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path_to_DS_3001_folder = '/content/drive/MyDrive/DS-3001/03_Tuning_Testing_Evaluating'
# path_to_DS_3001_folder = ''

# Update the path to your folder for the class
# Where you stored the data from the previous noteboook
os.chdir(path_to_DS_3001_folder)

## Section 1: Bias Variance Trade Off

- To illustrate the bias variance trade off, we'll use a synethetic data set that follows this formula:

$$
y_i = \sin(x_i)x_i + \varepsilon_i
$$

- This is a non-linear function with an irreducible noise component. We can use this data to fit models of varying complexity and understand how these models perform.


In [ ]:
# Create parameters for a data set
N = 1000 # Number of observations
mean_x = 7 # The mean for the data
se_x = 2 # The standard deviation for the data
se_eps = 2 # The standard deviation of the irreducible error

# Create the input data (x)
seed = 112 # Set a random seed for reproducibility
eps_seed = 10
x = np.random.default_rng(seed = seed).normal(mean_x, se_x, N).T # Generate x's
eps = np.random.default_rng(seed = eps_seed).normal(0, se_eps, N).T # Generate irreducible error

# Calculate the output data (y)
y_actual = np.sin(x)*x # Calculating using the function, the true value without error
y = y_actual + eps # Adding irreducible error

# Store in a dataframe
df = pd.DataFrame(
  {
    'y': y,
    'x': x,
    'y_actual': y_actual
  }
)

df = df.sort_values('x')
df.head()

**We can visualize what this function looks like with the noise added:**

In [ ]:
# We can plot the data

# Plot the data points in grey
plt.scatter(
    df['x'], df['y'],
    label='Observed',color='grey'
)

# Plot the true data in black
plt.plot(
    df['x'], df['y_actual'],
    label ='True', color='black'
)

# Add the required labels
plt.xlabel('X')
plt.ylabel('Y')
plt.legend(loc='lower left')
plt.show()

**We can look at fitting a model to these data with different degrees of polynomial expansion**.
  - Lower degree models represent more simplistic models. This will result in high bias and low model variance.
  - Higher degree models represent more complex models. This will decrease our bias and increase our model variance.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression # Import linear regression model

X = df.loc[:,['x'] ]
y = df.loc[:,['y'] ]

# Loop over degrees 1 to 9
rmse_on_data = []
for d in np.arange(1,10):
    # Create an expander object with degree d
    expander = PolynomialFeatures(degree=d,include_bias=False)

    # Expand the X variable to get degrees up to d
    Z = expander.fit_transform(X[['x']])
    names = expander.get_feature_names_out() # Get the names of these variables

    # Fit a regression using the degree
    model = LinearRegression(fit_intercept=True)
    model = model.fit(Z,y) # Run regression
    y_hat = model.predict(Z) # Get the preidctions
    rmse = np.sqrt( np.mean( (y - y_hat)**2 )) # Calculate the RMSE
    r2 = model.score(Z, y) # Calculate the R^2 values

    # Plot the model compared with the data
    plt.scatter(X['x'],y, label='Observed',color='grey')
    plt.plot(X['x'],y_hat, label ='Fit',color='black')
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.title(f'Comparison of the Fit Model (Degree: {d}) to the Data\nRMSE: {rmse:.2f}, R^2: {r2:.2f}')
    plt.legend(loc='lower left')
    plt.show()
    print('\n')

    # Append the rmse to visualize
    rmse_on_data.append(rmse)

In [ ]:
# Plot the RMSE across degrees for these examples above
plt.plot(np.arange(1,10),rmse_on_data, marker = 'o')
plt.xlabel('Degree')
plt.ylabel('Root Mean Squared Error (RMSE)')
plt.title('RMSE Across Degrees on Training Data')
plt.show()

**What if we make an incredibly complex model:** We'll treat the x variable as a categorical varaible. This means that for each possible value of x, we'll have a coefficient that tells us the mean value of y.

In [ ]:
# Get the one-hot encoding for the x variable
x_dummies = pd.get_dummies(X['x'],dtype=int)

# Fit a linear regression model for the result
model = LinearRegression(fit_intercept=False)
model = model.fit(x_dummies, y)

# Get the predictions
y_hat = model.predict(x_dummies)

# Calculate the root mean squared error
rmse = np.sqrt( np.mean( (y - y_hat)**2 ))

# Calculate the R^2 value
print('Rsq: ', model.score(x_dummies, y), ', RMSE: ', rmse) # R2

# Plot the predicted model results on the data
plt.plot(X['x'], y_hat, label ='Fit',color='black')
plt.scatter(X['x'], y, label='Observed',color='grey')
plt.xlabel("X")
plt.ylabel("Y")
plt.title('Extremely Overfit Model on Synthetic Data Set')
plt.legend(loc='lower left')
plt.show()

**This model is clearly overfit:** There is small bias now because each prediction is effectively the exact point from the training data set, but the variance is large.

**Let's compare the fitted model to the true underlying data generation process:**

In [ ]:
# List to hold values for the rmse on the actual underlying function
rmse_on_true_result = []

# Pull out the actual value that doesn't include the noise
y = df['y']
y_actual = df['y_actual']

# Loop over degrees that go from 1 to 24 by 3
for d in np.arange(1,25,3):
    # Create an expander object with degree d
    expander = PolynomialFeatures(degree=d,include_bias=False)

    # Expand the X variable to get degrees up to d
    Z = expander.fit_transform(X[['x']])
    names = expander.get_feature_names_out() # Get the names of these variables

    # Fit a regression using the degree
    model = LinearRegression(fit_intercept = True)
    model = model.fit(Z, y) # Run regression
    y_hat = model.predict(Z) # Get the preidctions
    rmse = np.sqrt( np.mean( (y_actual - y_hat)**2 )) # Calculate the RMSE
    r2 = model.score(Z, y_actual) # Calculate the R^2 values

    # Plot the model compared with the data
    plt.plot(X['x'], y_actual, label='True',color='grey')
    plt.plot(X['x'], y_hat, label ='Fit',color='red')
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.title(f'Comparison of the Fit Model (Degree: {d}) to the Underlying Data\nRMSE: {rmse:.2f}, R^2: {r2:.2f}')
    plt.legend(loc='lower left')
    plt.show()
    print('\n')

    # Append the rmse to visualize
    rmse_on_true_result.append(rmse)

## Section 2: K-Fold Cross Validation

* Next, we'll consider how our models perform across different degrees with different train test spilts.
* Previously, we only used one train/test split.
* Cross validation allows us to consider how our model would perform with different train/test splits.
* There's two ways we can perform K-Fold Cross Validation in Python:
  - Sklearn: More control
  - Sklearn: Less control

**Let's start with the more hands on approach:**

* We'll use the [KFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) function from Skelarn to split our data into K equally sized folds. We'll do tests on these.

In [ ]:
# We'll start by importing the KFold function from Sklearn
from sklearn.model_selection import KFold

In [ ]:
### We start with a fixed model that we want to investigate
# Let's start with degree 3
d = 3

# Create an expander object with degree d
expander = PolynomialFeatures(degree=d,include_bias=False)

# Expand the X variable to get degrees up to d
Z = expander.fit_transform(X[['x']])
names = expander.get_feature_names_out() # Get the names of these variables

### Creating the K-Splits
# We can create a K-Folds object using Sklearn
# We'll start with a case of K = 10, which is common
kf = KFold(n_splits = 10, shuffle = True, random_state = 42)

# Create a list to store the scores
rmse_by_k = []

# Split the observations of Z into K Folds
# Loop over the different folds
for train_index, test_index in kf.split(Z):
  # Isolate the design matrix for the train and test data
  X_train = Z[train_index]
  X_test = Z[test_index]

  # Isolate the y variable for the train and test data
  y_train = y.iloc[train_index]
  y_test = y.iloc[test_index]

  # Fit the regression model on this training data set
  model = LinearRegression(fit_intercept=True)
  model.fit(X_train, y_train) # Run regression
  y_hat = model.predict(X_test) # Get the preidctions on the test data set

  # Calculate the Mean Squared Error on the test data set
  rmse = np.sqrt( np.mean( (y_test - y_hat)**2 )) # Calculate the RMSE

  # Save the rmse for this Kth Fold
  rmse_by_k.append(rmse)

# We can look at the distribution of the RMSE across the different Fold's
sns.boxplot(np.array(rmse_by_k))
plt.xlabel('RMSE')
plt.ylabel('Frequency')
plt.show()

# Print the actual score, their mean and median, and std
print('\n')
print('RMSE for each Fold:', rmse_by_k)
print('Means RMSE across Folds:', np.mean(rmse_by_k))
print('Median RMSE across Folds:', np.median(rmse_by_k))
print('STD RMSE across Folds:', np.std(rmse_by_k))

**Let's try a more automatic approach:**

- We can use the [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) function to automatically do the for loop we included above.

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
# Create the model object
model = LinearRegression(fit_intercept=True)

# Create the KFold object with the number of splits
kf = KFold(n_splits = 10, shuffle = True, random_state = 42)

# Calculate the scores across K Folds using the cross_val_score function
scores = cross_val_score(
    model, Z, y,
    cv = kf, # The KFold object we defined,
    scoring = 'neg_root_mean_squared_error' # The negative RMSE is used as the calcualted scoring function
)

# We can take the negative to get back to the RMSE instead of negative rmse
scores = -scores

# We can look at the distribution of the RMSE across the different Fold's
sns.boxplot(np.array(scores))
plt.xlabel('RMSE')
plt.ylabel('Frequency')
plt.show()

# Print the actual score, their mean and median, and std
print('\n')
print('RMSE for each Fold:', scores)
print('Means RMSE across Folds:', np.mean(scores))
print('Median RMSE across Folds:', np.median(scores))
print('STD RMSE across Folds:', np.std(scores))

**Taking this one step more, we can consider various values for degree and do KFold Cross Validation for each**

- Investigate the results of this analysis if you increase the max degree that you look at.

In [ ]:
# Loop over the possible values of degree

mean_rmse_by_d = []
std_rmse_by_d = []

max_degree = 10
degrees = np.arange(1, max_degree + 1)

for d in degrees:
  # Create an expander object with degree d
  expander = PolynomialFeatures(degree=d, include_bias=False)

  # Expand the X variable to get degrees up to d
  Z = expander.fit_transform(X[['x']])
  names = expander.get_feature_names_out() # Get the names of these variables

  # Create the model object
  model = LinearRegression(fit_intercept=True)

  # Create the KFold object with the number of splits
  kf = KFold(n_splits = 10, shuffle = True, random_state = 42)

  # Calculate the scores across K Folds using the cross_val_score function
  scores = cross_val_score(
      model, Z, y,
      cv = kf, # The KFold object we defined,
      scoring = 'neg_root_mean_squared_error' # The negative RMSE is used as the calcualted scoring function
  )

  # We can take the negative to get back to the RMSE instead of negative rmse
  scores = -scores

  # Append the results
  mean_rmse_by_d.append(np.mean(scores))
  std_rmse_by_d.append(np.std(scores))

# Plot the results
mean_rmse_by_d = np.array(mean_rmse_by_d)
std_rmse_by_d = np.array(std_rmse_by_d)

plt.plot(degrees, mean_rmse_by_d, label = 'Mean RMSE', marker = 'o')
plt.fill_between(
    degrees,
    mean_rmse_by_d - std_rmse_by_d, # Lower Bound
    mean_rmse_by_d + std_rmse_by_d, # Upper Bound
    label = '1 Standard Deviation of RMSE',
    alpha = 0.2,
    color = 'grey'
)
plt.title('Comparison of Model Performance (RMSE) across Degrees\n with K-Fold Cross Validation')
plt.xlabel('Degree')
plt.ylabel('RMSE')
plt.legend()
plt.show()